In [ ]:
from ultralytics import YOLO
from LLM.prompt import LocalLLMPrompter
import json
import numpy as np
import os
import matplotlib.pyplot as plt
import random
from ai2thor.controller import Controller
from ai2thor.platform import CloudRendering
from dependencies.sim.action_primitives import *
from dependencies.sim.task_utils import TaskUtil, save_data, get_in_memory_episode_result
from dependencies.sim.constants import TASK_DICT

model = YOLO("yoloe-26x-seg.pt")
prompter = LocalLLMPrompter(think=True)

np.random.seed(42)
random.seed(42)

# Test 1: Simulated Data Changing Perception to YOLOE

In [ ]:
def flatten_list(values):
    output = []
    for item in values:
        if isinstance(item, list):
            output.extend(item)
        else:
            output.append(item)
    return output

def get_failure_injection_idx(task_util, actions, task, action_idxs, nav_idxs, interact_cnt=0, nav_cnt=0):
    counter = 0
    already_used = flatten_list([f[1] for f in task_util.failures_already_injected])
    while True:
        if task_util.chosen_failure in ("missing_step", "failed_action"):
            if task_util.chosen_failure == "missing_step" and "specified_missing_steps" in task:
                cnt = sum(1 for f in task_util.failures_already_injected if f[0] == "missing_step")
                if cnt < len(task["specified_missing_steps"]):
                    return task["specified_missing_steps"][cnt]

            failure_injection_idx = np.random.choice(action_idxs[interact_cnt:])
            if "toggle_off" in actions[failure_injection_idx] or "close_obj" in actions[failure_injection_idx]:
                counter += 1
                continue
            if not already_used or failure_injection_idx not in already_used:
                return int(failure_injection_idx)

        elif task_util.chosen_failure == "drop":
            failure_injection_idx = np.random.choice(nav_idxs[nav_cnt:])
            return int(failure_injection_idx)

        if counter > 60:
            return -1
        counter += 1

def convert_step_to_timestep(step, video_fps):
    seconds = step // video_fps
    return "{:02d}:{:02d}".format(int(seconds / 60), seconds % 60)

def normalize_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() == "true"
    return bool(value)

def resolve_folder_name(task):
    if "folder_name" in task and task["folder_name"]:
        return os.path.join(TASK_DICT[task["task_idx"]], task["folder_name"])

    if "specific_folder_name" in task and task["specific_folder_name"]:
        return task["specific_folder_name"]
    general = task.get("general_folder_name", "single-episode")
    return os.path.join(TASK_DICT[task["task_idx"]], general)

def run_single_sim_episode_in_memory(task_json_path, data_path="."):
    """Run exactly one simulated episode and return all outputs in memory."""
    with open(task_json_path, "r") as f:
        task = json.load(f)

    failure_injection = normalize_bool(task.get("failure_injection", False))
    folder_name = resolve_folder_name(task)

    controller = Controller(
        agentMode="default",
        massThreshold=None,
        scene=task["scene"],
        visibilityDistance=1.5,
        gridSize=0.25,
        renderDepthImage=True,
        renderInstanceSegmentation=True,
        width=960,
        height=960,
        fieldOfView=60,
        platform=CloudRendering,
    )

    try:
        reachable_positions = controller.step(action="GetReachablePositions").metadata["actionReturn"]
        chosen_failure = task.get("chosen_failure", None)
        failure_injection_params = task.get("failure_injection_params", {})

        task_util = TaskUtil(
            folder_name=folder_name,
            controller=controller,
            reachable_positions=reachable_positions,
            failure_injection=failure_injection,
            index=0,
            repo_path=data_path,
            chosen_failure=chosen_failure,
            failure_injection_params=failure_injection_params,
            in_memory=True,
        )

        if task_util.chosen_failure in ("blocking", "occupied", "occupied_put") and "failure_injection_params" in task:
            place_obj(task_util, task["failure_injection_params"])

        if task_util.chosen_failure == "wrong perception" and "disp_x" in failure_injection_params:
            place_obj(task_util, task["failure_injection_params"])

        if "preactions" in task:
            for preaction_instr in task["preactions"]:
                parts = [item.strip("() ") for item in preaction_instr.split(",")]
                preaction = parts[0]
                params = parts[1:]
                globals()[preaction](task_util, *params)

        instrs = list(task["actions"])
        executed_instrs = []
        action_idxs = []
        nav_idxs = []

        for idx, instr in enumerate(instrs):
            parts = [item.strip("() ") for item in instr.split(",")]
            action = parts[0]
            if action in task_util.INTERACT_ACTION_PRIMITIVES:
                action_idxs.append(idx)
            if action == "navigate_to_obj":
                nav_idxs.append(idx)

        failure_injection_idx = None
        if failure_injection:
            failure_injection_idx = get_failure_injection_idx(task_util, instrs, task, action_idxs, nav_idxs)
            if failure_injection_idx == -1:
                return {"error": "No valid failure injection index found for this episode."}

        nav_counter = 0
        interact_counter = 0

        for idx, instr in enumerate(instrs):
            parts = [item.strip("() ") for item in instr.split(",")]
            action = parts[0]
            params = parts[1:]
            func = globals()[action]

            if action in task_util.INTERACT_ACTION_PRIMITIVES:
                interact_counter += 1
            if action == "navigate_to_obj":
                nav_counter += 1

            to_drop = False
            if (
                failure_injection
                and not task_util.failure_added
                and task_util.chosen_failure == "drop"
                and idx == failure_injection_idx
            ):
                to_drop = True
                params.append(to_drop)
                params.append(failure_injection_idx)

            if (
                failure_injection
                and not task_util.failure_added
                and task_util.chosen_failure == "missing_step"
                and action in task_util.INTERACT_ACTION_PRIMITIVES
            ):
                if not isinstance(failure_injection_idx, list):
                    failure_injection_idx = [failure_injection_idx]
                if idx in failure_injection_idx:
                    if "gt_failure_reason" in task_util.gt_failure:
                        task_util.gt_failure["gt_failure_reason"] += ", " + instr
                    else:
                        task_util.gt_failure["gt_failure_reason"] = "Missing " + instr
                    task_util.gt_failure["gt_failure_step"] = task_util.counter + 1
                    if idx == failure_injection_idx[-1]:
                        task_util.failure_added = True
                        task_util.failures_already_injected.append([task_util.chosen_failure, failure_injection_idx])
                    else:
                        task_util.failure_added = False
                    continue

            fail_execution = False
            if (
                failure_injection
                and not task_util.failure_added
                and task_util.chosen_failure == "failed_action"
                and action in task_util.INTERACT_ACTION_PRIMITIVES
                and idx == failure_injection_idx
            ):
                fail_execution = True
                task_util.gt_failure["gt_failure_reason"] = "Failed to successfully execute " + instr
                task_util.gt_failure["gt_failure_step"] = task_util.counter + 1
                task_util.failures_already_injected.append([task_util.chosen_failure, failure_injection_idx])
                task_util.failure_added = True
                params.append(fail_execution)

            executed_instrs.append(instr)
            retval = func(task_util, *params)

            if failure_injection and retval is False:
                failure_injection_idx = get_failure_injection_idx(
                    task_util, instrs, task, action_idxs, nav_idxs,
                    interact_cnt=interact_counter, nav_cnt=nav_counter,
                )
                if failure_injection_idx == -1:
                    break

        for _ in range(2):
            e = controller.step(action="Done")
            save_data(task_util, e)

        if "gt_failure_reason" not in task_util.gt_failure:
            task_util.gt_failure["gt_failure_reason"] = "No failure added"
            task_util.gt_failure["gt_failure_step"] = 0

        updated_task = task.copy()
        updated_task["specific_folder_name"] = task_util.specific_folder_name
        updated_task["gt_failure_reason"] = task_util.gt_failure["gt_failure_reason"]
        updated_task["gt_failure_step"] = convert_step_to_timestep(
            task_util.gt_failure["gt_failure_step"], video_fps=1
        )
        updated_task["unity_name_map"] = task_util.unity_name_map
        updated_task["sounds"] = task_util.sounds
        updated_task["actions"] = executed_instrs

        result = get_in_memory_episode_result(task_util, task_metadata=updated_task)
        result["frame_count"] = len(result["frames"])
        return result
    finally:
        controller.stop()

episode_result = run_single_sim_episode_in_memory(
    task_json_path="data_test/sim/makeSalad-5/task.json",
    data_path=".",
)
print("Episode completed. Steps:", episode_result.get("step_count"))
print("Frames in memory:", episode_result.get("frame_count"))
episode_result